# Supplementary Fig. 3c: spike waveforms within bursts

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel c of Supplementary Fig. 3: mean waveform (with 95% confidence interval over bursts) of the first (black), second (red) and third (orange) spike of the bursts of three bursting units of the simulated Neuropixels evoked regime. Source data and svg are saved in figures/8_source_data/supp_fig3c/.

**method (as in the paper's notebook supp_fig3_CDEF_burst_modulation_of_spike_revised.ipynb)**
* recording: `dataset/00_raw/recording_npx_evoked` (1 hour, 20 kHz) converted to int16, band-pass filtered (300-3000 Hz), common median referenced and whitened
* bursts of a ground truth unit: at least 3 spikes with inter-spike intervals up to 21 ms and duration of at least 20 ms (`src/nodes/validation/burst_spikes.py`)
* for every burst, the waveform (2.5 ms) of its first three spikes on the site nearest to the unit, aligned on the negative peak; then the mean and the 95% confidence interval (1.96 x standard error) over the bursts

In [1]:
# SETUP PACKAGES
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
import spikeinterface as si
import spikeinterface.preprocessing as spre
warnings.simplefilter(action="ignore")

# SET PATHS
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)
from src.nodes.validation.burst_spikes import find_bursts_isi, get_burst_first_n_waveforms, ci95
REC_PATH = "dataset/00_raw/recording_npx_evoked"
GT_PATH = "dataset/00_raw/ground_truth_npx_evoked"

# PARAMETERS
FREQ_MIN, FREQ_MAX = 300, 3000      # band-pass filter (Hz)
MIN_SPIKES = 3                      # minimum number of spikes in a burst
ISI_THRESH_SEC = 0.021              # maximum inter-spike interval within a burst
NUM_SPIKES = 3                      # first spikes of each burst
WINDOW_MS = 2.5                     # waveform duration
UNITS = ["2281009", "2873667", "506676"]     # bursting units shown in the paper
SPIKE_NAMES = ["1st spike", "2nd spike", "3rd spike"]
LINE_COLORS = ["k", "r", "orange"]
AREA_COLORS = ["grey", "pink", "orange"]

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

## Load the recording and the ground truth, find the bursts

In [2]:
%%time

# recording: preprocessed lazily (only the traces around the spikes are computed)
Recording = si.load_extractor(REC_PATH)
Recording = spre.astype(Recording, "int16")
Recording = spre.bandpass_filter(Recording, freq_min=FREQ_MIN, freq_max=FREQ_MAX)
Recording = spre.common_reference(Recording, reference="global", operator="median")
Recording = spre.whiten(Recording, dtype="float32")
sfreq = Recording.get_sampling_frequency()
window_frames = int(WINDOW_MS / (1 / sfreq) / 1000)

# ground truth: bursts of each unit
Gtruth = si.load_extractor(GT_PATH)
bursts = {u: find_bursts_isi(Gtruth.get_unit_spike_train(u) / sfreq, isi_threshold=ISI_THRESH_SEC, min_spikes=MIN_SPIKES) for u in Gtruth.unit_ids}
bursting_units = [u for u in Gtruth.unit_ids if len(bursts[u]) > 0]
print("bursting units (number of bursts):", [(u, len(bursts[u])) for u in bursting_units])

bursting units (number of bursts): [('506676', 68), ('2281009', 2), ('2873667', 2), ('3972082', 1)]
CPU times: user 14.4 s, sys: 264 ms, total: 14.6 s
Wall time: 13.9 s


## Mean waveform and 95% confidence interval of the first three spikes of the bursts

In [3]:
%%time

rows = []
for unit in UNITS:
    waves = get_burst_first_n_waveforms(Recording, Gtruth, bursts, unit, sfreq, peak_pos=int(window_frames / 2), window=window_frames, frames_before=100, frames_after=100, num_spikes=NUM_SPIKES)
    mean_waves, ci95_waves = waves.mean(axis=0), ci95(waves)            # (spikes, frames)
    for s in range(NUM_SPIKES):
        rows.append(pd.DataFrame({
            "unit": unit, "mtype": Gtruth.get_property("mtype", [unit])[0], "etype": Gtruth.get_property("etype", [unit])[0], "n_bursts": waves.shape[0],
            "spike": SPIKE_NAMES[s], "time_ms": (np.arange(window_frames) - window_frames // 2) / sfreq * 1000, "voltage_mean_uV": mean_waves[s], "voltage_ci95_uV": ci95_waves[s]}))
df = pd.concat(rows, ignore_index=True)
display(df.drop_duplicates("unit")[["unit", "mtype", "etype", "n_bursts"]])

,unit,mtype,etype,n_bursts
0,2281009,L4_UPC,cADpyr,2
150,2873667,L4_LBC,cACint,2
300,506676,L23_SBC,cACint,62


CPU times: user 2.36 s, sys: 8.99 ms, total: 2.37 s
Wall time: 595 ms


## Plot, same as the paper's panel c

In [4]:
# paper arrangement (paper/figures/supp_fig3.pdf) rescaled to a figure of at most 185 mm, Arial 5 pt text, 7 pt titles, editable (src/nodes/validation/burst_plots.py)
from src.nodes.validation.burst_plots import plot_bursts

def plot_source_data(df: pd.DataFrame):
    return plot_bursts(df)

## Plot and save the figure and the source data

In [5]:
fig = plot_source_data(df)
os.makedirs("figures/8_source_data/supp_fig3c", exist_ok=True)
plt.savefig("figures/8_source_data/supp_fig3c/supp_fig3c.svg", transparent=True)   # the .pdf is written next to the .svg
df.to_excel("figures/8_source_data/supp_fig3c/supp_fig3c.xlsx", index=False)

## Check: replot from the saved source data

In [6]:
df = pd.read_excel("figures/8_source_data/supp_fig3c/supp_fig3c.xlsx", dtype={"unit": str})
fig = plot_source_data(df)